In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

from myjive.fem import NodeSet, ElementSet

from experiments.reproduction.nonhierarchical.frp_dic import caching, params
from experiments.reproduction.nonhierarchical.frp_dic import misc

n_fiber = params.geometry_params["n_fiber"]
h = 0.100
fibers = caching.get_or_calc_fibers()
rve_size = params.geometry_params["rve_size"]
r_fiber = params.geometry_params["r_fiber"]

h_str = "{:.3f}".format(h)
mesh_sizes = [h, h_str + "r1", h_str + "d1", h_str + "d2", h_str + "h1", h_str + "h2"]
meshes = []

for mesh_size in mesh_sizes:
    if isinstance(mesh_size, float):
        mesh = caching.get_or_calc_mesh(h=mesh_size)
    elif isinstance(mesh_size, str):
        if "r" in mesh_size:
            mesh = caching.get_or_calc_mesh(h=mesh_size)
        elif "d" in mesh_size:
            mesh = caching.get_or_calc_dual_mesh(h=mesh_size)
        elif "h" in mesh_size:
            mesh = caching.get_or_calc_hyper_mesh(h=mesh_size, do_groups=True)
        else:
            assert False
    else:
        assert False

    meshes.append(mesh)

In [ ]:
def get_edges(mesh):
    if isinstance(mesh, ElementSet):
        do_groups = False
        elems = mesh
        nodes = elems.get_nodes()
    elif isinstance(mesh, tuple):
        do_groups = True
        nodes, elems, egroups = mesh

    assert isinstance(nodes, NodeSet)
    assert isinstance(elems, ElementSet)

    if do_groups:
        fiber_edges = set()
        matrix_edges = set()

        for ielem in egroups["fiber"].get_indices():
            inodes = elems[ielem]
            for i, j in [(0, 1), (1, 2), (2, 0)]:
                if inodes[i] < inodes[j]:
                    edge = (inodes[i], inodes[j])
                else:
                    edge = (inodes[j], inodes[i])

                fiber_edges.add(edge)

        for ielem in egroups["matrix"].get_indices():
            inodes = elems[ielem]
            for i, j in [(0, 1), (1, 2), (2, 0)]:
                if inodes[i] < inodes[j]:
                    edge = (inodes[i], inodes[j])
                else:
                    edge = (inodes[j], inodes[i])

                if edge not in fiber_edges:
                    matrix_edges.add(edge)

        return fiber_edges, matrix_edges

    else:
        edges = set()

        for ielem, inodes in enumerate(elems):
            for i, j in [(0, 1), (1, 2), (2, 0)]:
                if inodes[i] < inodes[j]:
                    edge = (inodes[i], inodes[j])
                else:
                    edge = (inodes[j], inodes[i])

                edges.add(edge)

        return edges

In [ ]:
def plot_mesh(mesh1, mesh2=None, *, fname=None):
    if isinstance(mesh1, ElementSet):
        elems1 = mesh1
        nodes1 = elems1.get_nodes()
    else:
        nodes1, elems1, egroups1 = mesh1

    edges1 = get_edges(elems1)

    if mesh2 is not None:
        if isinstance(mesh2, ElementSet):
            elems2 = mesh2
            nodes2 = elems2.get_nodes()
        else:
            nodes2, elems2, egroups2 = mesh2

        edges2 = get_edges(elems2)

    fig, ax = plt.subplots()

    for ielem in egroups1["fiber"].get_indices():
        inodes = elems1[ielem]
        coords = nodes1[inodes]
        patch = Polygon(coords, color="0.8")
        ax.add_patch(patch)

    for edge in edges1:
        coords = nodes1[list(edge)]
        ax.plot(coords[:, 0], coords[:, 1], color="k", linewidth=0.5)

    if mesh2 is not None:
        for edge in edges2:
            coords = nodes2[list(edge)]
            ax.plot(coords[:, 0], coords[:, 1], color="0.4", linewidth=0.5)

    ax.set_xlim((-0.4, 0.4))
    ax.set_ylim((-0.4, 0.4))
    ax.set_aspect("equal")
    ax.set_axis_off()

    if fname is not None:
        plt.savefig(fname=fname, bbox_inches="tight")

    plt.show()

In [ ]:
for mesh_size, mesh in zip(mesh_sizes, meshes):
    if isinstance(mesh_size, float):
        fname = "rve-mesh_h-{:.3f}.pdf".format(mesh_size)
        obs_mesh = None
    elif isinstance(mesh_size, str):
        fname = "rve-mesh_h-{}.pdf".format(mesh_size)
        obs_mesh = meshes[0] if "d" in mesh_size else None
    else:
        assert False

    fname = os.path.join("plots", fname)
    plot_mesh(mesh, obs_mesh, fname=fname)